# derived_8.4-hybrid-lstm-1.0 — Hybrid LSTM Context Vector (ctx) + XGBoost Evaluation

This experiment evaluates the **Hybrid LSTM + XGBoost** modeling architecture on the  Washington-only dataset split (7 stations, 2023–2025 test set).

### Architecture & Pipeline Overview
1. **Phase 1 (BiLSTM Training)**: Train  v9 on sequence dataset until early-stopping convergence on validation RMSE.
2. **Phase 2 (Frozen CTX Extraction)**: Freeze  (, ) and extract 160-dimensional attention-pooled hidden state vectors (..).
3. **Phase 3 (XGBoost Hybrid Fusion)**: Concatenate  with tabular features (54 shared global backbone + cluster add-on deltas) to train and evaluate XGBoost models against the pure tabular baselines from .

In [1]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import yaml

# Set up absolute paths for project root and experiment directory
EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.0").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.0").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

print(f"[Setup] Project Root: {PROJECT_ROOT}")
print(f"[Setup] Experiment Directory: {EXP_DIR}")


[Setup] Project Root: /scratch/user/u.rp352032/MDR-Project
[Setup] Experiment Directory: /scratch/user/u.rp352032/MDR-Project/notebooks/experiment/derived_8.4-hybrid-lstm-1.0


## Phase 1 & Phase 2: BiLSTM Model Training & Frozen `ctx` Extraction

In this section, we train the local `BiLSTMAttn` model on the `derived_8.4` sequence dataset until early-stopping convergence. Once trained, we freeze the model (`model.eval()`, `torch.no_grad()`) and extract 160-dimensional attention-pooled context vectors (`ctx`) across all train, val, and test split samples.


In [2]:
import sys, json, numpy as np, pandas as pd
from pathlib import Path
from lstm.train import train_lstm_and_extract_ctx

EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.0").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.0").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

data_dir = PROJECT_ROOT / "data/splits/derived_8.4"
artifacts_dir = EXP_DIR / "artifacts"

# Load pre-computed frozen CTX representations if available, or train & extract
if (artifacts_dir / "ctx_test.npy").exists() and (artifacts_dir / "lstm_metrics.json").exists():
    print("[LSTM] Loading pre-extracted frozen CTX representations from artifacts...")
    ctx_tr = np.load(artifacts_dir / "ctx_train.npy")
    ctx_va = np.load(artifacts_dir / "ctx_val.npy")
    ctx_te = np.load(artifacts_dir / "ctx_test.npy")
    with open(artifacts_dir / "lstm_metrics.json") as f:
        lstm_metrics = json.load(f)
else:
    ctx_tr, ctx_va, ctx_te, lstm_metrics = train_lstm_and_extract_ctx(data_dir, artifacts_dir)

print(f"\n[LSTM Phase Complete] Train CTX: {ctx_tr.shape}, Val CTX: {ctx_va.shape}, Test CTX: {ctx_te.shape}")
print(f"[LSTM Test Performance] R2 = {lstm_metrics['test']['r2']:.4f}, RMSE = {lstm_metrics['test']['rmse']:.5f}")


[LSTM] Loading pre-extracted frozen CTX representations from artifacts...

[LSTM Phase Complete] Train CTX: (9803, 160), Val CTX: (4805, 160), Test CTX: (6620, 160)
[LSTM Test Performance] R2 = 0.6186, RMSE = 0.06291


## Phase 3: XGBoost Hybrid Modeling & Evaluation

We evaluate four models on the `derived_8.4` test set (6,620 samples):
1. **Global Single Model (54 Backbone)** — Pure tabular baseline
2. **Clustering_V0_Full_k2 (Winner c0=0, c1=10)** — Pure tabular MoE baseline
3. **Global Single Model (54 Backbone + 160 CTX)** — Hybrid global model (214 input features)
4. **Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)** — Hybrid MoE model (214/224 input features)


In [3]:
import sys, json, yaml, numpy as np, pandas as pd
from pathlib import Path

EXP_DIR = Path("experiment/derived_8.4-hybrid-lstm-1.0").resolve() if Path("experiment/derived_8.4-hybrid-lstm-1.0").exists() else Path(".").resolve()
PROJECT_ROOT = EXP_DIR.parents[2]
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(EXP_DIR))

from eval_hybrid.data import load_hybrid_experiment_data
from eval_hybrid.evaluator import HybridStrategyEvaluator
from run_eval import compute_c1_gain_additions

artifacts_dir = EXP_DIR / "artifacts"
models_dir = EXP_DIR / "models"

with open(EXP_DIR / "config.yaml") as f:
    config = yaml.safe_load(f)

# Load tabular features + concatenated 160-dim CTX representations
data = load_hybrid_experiment_data(PROJECT_ROOT, EXP_DIR, config)
c1_additions = compute_c1_gain_additions(data, config)

summary_records = []

# 1. Global Single Baseline
eval_global = HybridStrategyEvaluator(data, config, "Global_Single", models_dir=models_dir)
res_g_base = eval_global.fit_and_evaluate("Global Single Model (54 Backbone)", "Global_Single_54_Backbone", data.shared_backbone_54)
summary_records.append(res_g_base.as_record())

# 2. Clustering_V0_Full_k2 Baseline
eval_v0 = HybridStrategyEvaluator(data, config, "Clustering_V0_Full_k2", models_dir=models_dir)
res_v0_base = eval_v0.fit_and_evaluate("Clustering_V0_Full_k2 (Winner c0=0, c1=10)", "Clustering_V0_Full_k2_c0_0_c1_10", data.shared_backbone_54, {"0": [], "1": c1_additions})
summary_records.append(res_v0_base.as_record())

# 3. Global Single Hybrid
res_g_hybrid = eval_global.fit_and_evaluate("Global Single Model (54 Backbone + 160 CTX)", "Global_Single_54_Backbone_160_CTX", data.hybrid_backbone_214)
summary_records.append(res_g_hybrid.as_record())

# 4. Clustering_V0_Full_k2 Hybrid
res_v0_hybrid = eval_v0.fit_and_evaluate("Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)", "Clustering_V0_Full_k2_c0_0_c1_10_160_CTX", data.hybrid_backbone_214, {"0": [], "1": c1_additions})
summary_records.append(res_v0_hybrid.as_record())

df_summary = pd.DataFrame(summary_records).sort_values("pooled_r2", ascending=False)
df_summary.to_csv(artifacts_dir / "summary_records.csv", index=False)

print("\n" + "=" * 75)
print("FINAL HYBRID LEADERBOARD (derived_8.4-hybrid-lstm-1.0)")
print("=" * 75)
print(df_summary[["model_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]].to_string(index=False))


/scratch/user/u.rp352032/MDR-Project/notebooks/.venv/lib64/python3.12/site-packages/xgboost/core.py:751: UserWarning: [19:09:26] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



FINAL HYBRID LEADERBOARD (derived_8.4-hybrid-lstm-1.0)
                                          model_name  pooled_r2  pooled_rmse  pooled_ubrmse  pooled_bias  pooled_mae  pooled_pearson
          Clustering_V0_Full_k2 (Winner c0=0, c1=10)   0.814960     0.043820       0.043337     0.006486    0.033719        0.905594
                   Global Single Model (54 Backbone)   0.779230     0.047864       0.046687     0.010548    0.037059        0.889432
Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 160 CTX)   0.772048     0.048636       0.047864     0.008633    0.036382        0.886389
         Global Single Model (54 Backbone + 160 CTX)   0.760117     0.049893       0.048944     0.009683    0.037291        0.880724


## Results & Diagnostic Summary

We display the leaderboard comparison table showing Pooled $R^2$, RMSE, ubRMSE, Bias, MAE, and Pearson correlation for all four evaluated models.


In [4]:
# Display styled summary dataframe
df_summary[["model_name", "pooled_r2", "pooled_rmse", "pooled_ubrmse", "pooled_bias", "pooled_mae", "pooled_pearson"]]


,model_name,pooled_r2,pooled_rmse,pooled_ubrmse,pooled_bias,pooled_mae,pooled_pearson
1,"Clustering_V0_Full_k2 (Winner c0=0, c1=10)",0.814960,0.043820,0.043337,0.006486,0.033719,0.905594
0,Global Single Model (54 Backbone),0.779230,0.047864,0.046687,0.010548,0.037059,0.889432
3,"Clustering_V0_Full_k2 (Winner c0=0, c1=10 + 16...",0.772048,0.048636,0.047864,0.008633,0.036382,0.886389
2,Global Single Model (54 Backbone + 160 CTX),0.760117,0.049893,0.048944,0.009683,0.037291,0.880724
